In [2]:
import pandas as pd
import geopandas as gpd
import altair as alt
import json
from shapely.geometry import shape, Polygon
from shapely import wkt
from shapely.ops import linemerge, unary_union
alt.data_transformers.disable_max_rows()
import h3

In [3]:
stop_code_map = {
    # L2
    212:125, # universitatm
    218:131, # clot
    # L3
    321:122, # espanya
    323:210, # paralel
    326:126, # pl cat
    327:213, # pg gr
    # L4
    413:221, # la pau
    424:127, # uriquinaona
    425:213, # pg gracia
    434:339, # trinitat nova
    # L5
    517:120, #pl sants
    518:319, # sants
    521:328, # diagonal
    522:427,  # verdaguer
    523:427, # sagrada fam
    526:133, # sagrera
    528:431, # maragall
    534:333 # vall d'hebron

}

In [4]:
with open('Data/Metro/metro_trajectories.json') as f:
    trajectories = json.load(f)


df_trajectories = []
for trajectory in trajectories['features']:
    properties = trajectory.get('properties', {})
    tram = properties.get('NOM_TRAM_LINIA')
    nom_linia = properties.get('NOM_LINIA')
    origen = properties.get('CODI_ESTACIO_INI')
    dest = properties.get('CODI_ESTACIO_FI')                   
    df_trajectories.append({
            'origen': origen,
            'dest': dest,
            'tram': tram,
            'linia': nom_linia,
            'type': 'Metro',
            'geometry': shape(trajectory['geometry'])
        })
df_trajectories_all = pd.DataFrame(df_trajectories)
df_trajectories_all['dest'] = df_trajectories_all['dest'].replace(stop_code_map)
df_trajectories_all['origen'] = df_trajectories_all['origen'].replace(stop_code_map)  
geo_df_trajectories_all = gpd.GeoDataFrame(df_trajectories_all, geometry='geometry', crs=4326)
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Inici') == False]
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Final') == False]
geo_df_trajectories_all.to_crs('EPSG:25831', inplace=True)
geo_df_trajectories_all['length'] = geo_df_trajectories_all['geometry'].length 
geo_df_trajectories_all['speed'] = 25 /3.6 # 25 kmh to ms like in the paper
geo_df_trajectories_all['time'] = (geo_df_trajectories_all['length'] / geo_df_trajectories_all['speed']) / 60
# geo_df_trajectories_all['time'] = pd.to_timedelta(
#     geo_df_trajectories_all['length'] / geo_df_trajectories_all['speed'],
#     unit='s'
# )
# geo_df_trajectories_all['time'] = geo_df_trajectories_all['time'].apply(
#     lambda x: f"{int(x.total_seconds() // 60):02d}:{int(x.total_seconds() % 60):02d}"
# )
geo_df_trajectories_all['directed'] = False
geo_df_trajectories_all.to_crs('EPSG:4326', inplace=True)
geo_df_trajectories_all['directed'] = False
geo_df_trajectories_all = geo_df_trajectories_all[['origen', 'dest', 'tram', 'linia', 'type', 'length', 'speed', 'time', 'directed','geometry']]
geo_df_trajectories_all['origen'] = 'M' + '-' + geo_df_trajectories_all['linia'] + '-' + geo_df_trajectories_all['origen'].astype(str)
geo_df_trajectories_all['dest'] = 'M' + '-' + geo_df_trajectories_all['linia'] + '-' + geo_df_trajectories_all['dest'].astype(str)
geo_df_trajectories_all.to_crs('EPSG:4326', inplace=True)
excluded = ['L9N','L9S','L10N','L10S','L11','FM','TM']
geo_df_trajectories_all = geo_df_trajectories_all[~geo_df_trajectories_all['linia'].isin(excluded)]

In [5]:
transport_weight = 1
geo_df_trajectories_all.insert(9, 'cost', geo_df_trajectories_all['time'] * transport_weight)


In [ ]:
traj

In [1]:
geo_df_trajectories_all

NameError: name 'geo_df_trajectories_all' is not defined

In [15]:
geo_df_trajectories_all.to_csv("Edges/E-Metro.csv", index=False)